In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (FILTRO AGRESSIVO)")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

# =============================================================================
# 2. A EXCLUSÃO CIRÚRGICA (BUSCA POR FRAGMENTOS)
# =============================================================================
PACIENTES_REMOVIDOS = ['TEA_L_17', 'TEA_L_24', 'TEA_L_46', 'control_35']

# Função agressiva: se o nome do paciente estiver DENTRO da string do ID, ele marca como True
def deve_remover(id_csv):
    id_csv_str = str(id_csv).lower()
    for alvo in PACIENTES_REMOVIDOS:
        if alvo.lower() in id_csv_str:
            return True # Achou o suspeito
    return False # Paciente inocente

# Aplica a função e cria o dataframe limpo
df_agrupado['Marcado_Para_Exclusao'] = df_agrupado['ID'].apply(deve_remover)
df_limpo = df_agrupado[df_agrupado['Marcado_Para_Exclusao'] == False].copy()

print(f"Pacientes originais: {len(df_agrupado)}")
print(f"Pacientes após corte: {len(df_limpo)}")
# =============================================================================

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_limpo[features_ouro].values
y = df_limpo['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. O LOOCV COM XGBOOST
loo = LeaveOneOut()
valores_reais = []
predicoes_finais = []

modelo = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)[0]
    
    valores_reais.append(y_test[0])
    predicoes_finais.append(y_pred)

# 4. RESULTADOS
acuracia = accuracy_score(valores_reais, predicoes_finais)
acertos = sum(1 for r, p in zip(valores_reais, predicoes_finais) if r == p)
erros = len(valores_reais) - acertos

print(f"\n🎯 RESULTADO DO TESTE (N={len(df_limpo)}):")
print(f"Acurácia: {acuracia * 100:.2f}%")
print(f"Acertos:  {acertos} de {len(df_limpo)}")
print(f"Erros:    {erros}")

print("\n📊 RELATÓRIO DE CLASSIFICAÇÃO:")
print(classification_report(valores_reais, predicoes_finais, target_names=['Controle (0)', 'TEA (1)']))
print("="*80)

TESTE EXPLORATÓRIO: LOOCV COM EXCLUSÃO DE SUSPEITOS (FILTRO AGRESSIVO)
Pacientes originais: 42
Pacientes após corte: 38

🎯 RESULTADO DO TESTE (N=38):
Acurácia: 92.11%
Acertos:  35 de 38
Erros:    3

📊 RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

Controle (0)       0.92      0.96      0.94        23
     TEA (1)       0.93      0.87      0.90        15

    accuracy                           0.92        38
   macro avg       0.92      0.91      0.92        38
weighted avg       0.92      0.92      0.92        38



In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import xgboost as xgb

print("="*80)
print("A ILUSÃO DO GRID SEARCH: OVERFITTING DE HIPERPARÂMETROS NO K-FOLD")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_agrupado[features_ouro].values
y = df_agrupado['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. CONFIGURANDO A ARMADILHA DO GRID SEARCH
# Usaremos 10-Folds (aumenta a variância, facilitando achar um pico falso de acurácia)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Grade insana para o SVM (forçando ele a decorar os dados)
param_grid_svm = {
    'C': [0.1, 1, 10, 50, 100, 500, 1000],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 50, 100],
    'kernel': ['rbf']
}

# Grade insana para o XGBoost
param_grid_xgb = {
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.2, 0.3],
    'n_estimators': [50, 100, 200],
    'gamma': [0, 0.1, 0.5]
}

print("Iniciando varredura massiva de hiperparâmetros (Isso levará cerca de 1 minuto)...\n")

# Buscando com SVM
grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm, cv=cv, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_scaled, y)

# Buscando com XGBoost
grid_xgb = GridSearchCV(xgb.XGBClassifier(eval_metric='logloss', random_state=42), param_grid_xgb, cv=cv, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_scaled, y)

print("🏆 RESULTADOS DA ILUSÃO DO GRID SEARCH 🏆")
print(f"Melhor Acurácia SVM (10-Fold CV):     {grid_svm.best_score_ * 100:.2f}%")
print(f"Melhores parâmetros SVM:              {grid_svm.best_params_}")
print("-" * 50)
print(f"Melhor Acurácia XGBoost (10-Fold CV): {grid_xgb.best_score_ * 100:.2f}%")
print(f"Melhores parâmetros XGBoost:          {grid_xgb.best_params_}")
print("="*80)

A ILUSÃO DO GRID SEARCH: OVERFITTING DE HIPERPARÂMETROS NO K-FOLD
Iniciando varredura massiva de hiperparâmetros (Isso levará cerca de 1 minuto)...

🏆 RESULTADOS DA ILUSÃO DO GRID SEARCH 🏆
Melhor Acurácia SVM (10-Fold CV):     77.50%
Melhores parâmetros SVM:              {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
--------------------------------------------------
Melhor Acurácia XGBoost (10-Fold CV): 77.50%
Melhores parâmetros XGBoost:          {'gamma': 0, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

print("="*80)
print("A ILUSÃO DO VAZAMENTO DE DADOS: SMOTE ANTES DO K-FOLD")
print("="*80)

# 1. PREPARAÇÃO DOS DADOS
DIR_REPORTS = '../reports/'
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

df = pd.read_csv(caminho_arquivo)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

features_numericas = [c for c in df.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']]
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']
X = df_agrupado[features_ouro].values
y = df_agrupado['Target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. O CRIME METODOLÓGICO: SMOTE ANTES DO SPLIT
# Equalizamos a base para 24 vs 24
smote = SMOTE(random_state=42)
X_vazado, y_vazado = smote.fit_resample(X_scaled, y)

print(f"Pacientes Originais: {len(y)} (Controle: {sum(y==0)}, TEA: {sum(y==1)})")
print(f"Pacientes com SMOTE: {len(y_vazado)} (Controle: {sum(y_vazado==0)}, TEA: {sum(y_vazado==1)})\n")

# 3. K-FOLD CROSS-VALIDATION
# Agora aplicamos um K-Fold altamente respeitado na literatura
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
modelo = RandomForestClassifier(random_state=42)

# Rodamos a validação nos dados que já foram corrompidos pelo SMOTE
scores = cross_val_score(modelo, X_vazado, y_vazado, cv=cv, scoring='accuracy')

print("🏆 RESULTADO DA ILUSÃO (SMOTE + 10-Fold CV) 🏆")
print(f"Acurácia Média: {scores.mean() * 100:.2f}%")
print(f"Desvio Padrão:  {scores.std() * 100:.2f}%")
print(f"Acurácias por fold: {[round(s*100, 2) for s in scores]}")
print("="*80)

A ILUSÃO DO VAZAMENTO DE DADOS: SMOTE ANTES DO K-FOLD
Pacientes Originais: 42 (Controle: 24, TEA: 18)
Pacientes com SMOTE: 48 (Controle: 24, TEA: 24)

🏆 RESULTADO DA ILUSÃO (SMOTE + 10-Fold CV) 🏆
Acurácia Média: 84.00%
Desvio Padrão:  14.97%
Acurácias por fold: [np.float64(80.0), np.float64(60.0), np.float64(100.0), np.float64(80.0), np.float64(60.0), np.float64(80.0), np.float64(100.0), np.float64(80.0), np.float64(100.0), np.float64(100.0)]
